In [ ]:
"""
synth_curv_dist_plots.ipynb
Author: Henry Yuan
Description: Figure 3 of paper. Plot distributions of maximum curvature for different synthetic grains.
"""

# imports
import numpy as np
import sys
import matplotlib.pyplot as plt
sys.path.append('../../src')
import roundness_3d as r3d
import synthetic_grains as sg
import meshes
from format_plots import format_plots

CR, CS, CE, CL, CRD, CED, CWAD, CSTD = format_plots() # format plots and collect colors

In [ ]:
# 4 grains: euhdral with aspect = 2, semi-round with aspect = 2, round with aspect = 2, and sphere
aspects = np.array([2,2,2,1])
target_vertex_count = 5000 # constant for all grains
Krad = 2 # k-ring radius for curvature computation; same as in r3d.std_curvature()

fig, axes = plt.subplots(4, 1, figsize=(2.75, 5.5), sharex=True, sharey=False)
axes = axes.flatten()
axes[3].set_xlabel(r'$\kappa_{max}$ ($\text{voxel}^{-1}$)')
bin_size = 0.05
bins = np.arange(-bin_size, 0.75, bin_size) # just from looking at data; if new data, let it set default for new data and then reset bins

GRAY = "#A5A5A5"
cs = [CE,CS,CR,GRAY]

for i,roundness in enumerate([0,0.5,1,1]):
    aspect = aspects[i]

    # make synthetic grain
    bv, mesh, nv = sg.synthetic_zircon(target_vertex_count,aspect, roundness, rotate=False)

    # calculate curvatures
    verts = mesh.vertices
    faces = mesh.faces
    max_curvs = meshes.compute_curvatures(verts, faces, Krad=Krad, curv_type='max')
    vert_areas = meshes.compute_vertex_areas(verts,faces)

    # print max and min values to adjust histogram bins
    print(max_curvs.max())
    print(max_curvs.min())
    
    axes[i].hist(max_curvs, bins, color=cs[i], edgecolor=CL, linewidth=0.5)
    axes[i].set_ylabel('Frequency')

    # calculate weighted standard deviation of curvature
    std_curv = r3d.weighted_std(max_curvs, vert_areas)

    if i == 0:
        axes[i].plot([],[],label=f'Euhedral\n'+r'$\sigma=$'+f'{np.round(std_curv,2)} \nN = {nv}')
    elif i == 1:
        axes[i].plot([],[],label=f'Semi-round\n'+r'$\sigma=$' + f'{np.round(std_curv,2)} \nN = {nv}')
    else:
        axes[i].plot([],[],label=f'Round\n'+r'$\sigma=$' + f'{np.round(std_curv,2)} \nN = {nv}')
    axes[i].legend()
plt.tight_layout()
plt.savefig('fig3.pdf')